# 07 - Versioning, Testing Strategy, and CLI Operations

## Definition
Model versioning tracks lineage of artifacts and enables safe promotion/rollback.

## Theory
Registry metadata (metrics, hashes, parent version) provides auditability.

## Motivation
When production quality degrades, rollback must be explicit and fast.


In [1]:
from pathlib import Path
import os

CWD = Path.cwd()
ROOT = CWD if (CWD / "pyproject.toml").exists() else CWD.parent
os.chdir(ROOT)
os.environ.setdefault("MPLCONFIGDIR", str(ROOT / ".mplconfig"))
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)
Path("outputs/figures").mkdir(parents=True, exist_ok=True)
Path("outputs/benchmarks").mkdir(parents=True, exist_ok=True)


In [2]:
import json
from pathlib import Path

registry_path = Path("models/registry.json")
registry = json.loads(registry_path.read_text(encoding="utf-8"))
registry


{'schema_version': 1,
 'active_version': 'v2',
 'versions': {'v1': {'version_id': 'v1',
   'model_path': 'models/iris_model_v1.pkl',
   'metrics': {'accuracy': 0.9666666666666667,
    'precision_macro': 0.9696969696969697,
    'recall_macro': 0.9666666666666667,
    'f1_macro': 0.9665831244778612,
    'roc_auc_ovr': 1.0,
    'train_samples': 120,
    'test_samples': 30,
    'n_features': 4,
    'n_classes': 3,
    'seed': 42},
   'description': 'Baseline logistic regression package artifact',
   'author': 'train_model.py',
   'created_at': '2026-06-24T22:23:07.897912+00:00',
   'status': 'archived',
   'artifact_sha256': '4f0ff1b7f2217611dc619f675c5389056341c0071c2353a4efa6aa551b78e2f2',
   'dataset_fingerprint': '6e592aace77bce225cd6af938de731a5798c8c30ee9cc5f1cd6abdea81e3bb25',
   'parent_version': None,
   'tags': ['iris', 'baseline', 'v1']},
  'v2': {'version_id': 'v2',
   'model_path': 'models/iris_model.pkl',
   'metrics': {'model_name': 'KNN',
    'accuracy': 1.0,
    'precision

## Version Story in This Project
- `v1`: LogisticRegression baseline
- `v2`: benchmark-selected improved model

### Best Practice
Always keep parent-child linkage for traceability.


In [3]:
from ml_package.versioning import VersionRegistry

tmp_registry = Path("outputs/benchmarks/notebook07_registry.json")
if tmp_registry.exists():
    tmp_registry.unlink()

vr = VersionRegistry(str(tmp_registry))
vr.register("v1", "models/iris_model_v1.pkl", metrics={"f1_macro": 0.96}, description="baseline", allow_overwrite=True)
vr.activate("v1")
vr.register("v2", "models/iris_model_v2.pkl", metrics={"f1_macro": 1.00}, description="improved", parent_version="v1", allow_overwrite=True)
vr.activate("v2")
vr.rollback_to("v1")
vr.list_versions()


[{'version_id': 'v2',
  'model_path': 'models/iris_model_v2.pkl',
  'metrics': {'f1_macro': 1.0},
  'description': 'improved',
  'author': 'ml_package',
  'created_at': '2026-06-24T22:23:53.828491+00:00',
  'status': 'archived',
  'artifact_sha256': None,
  'dataset_fingerprint': None,
  'parent_version': 'v1',
  'tags': []},
 {'version_id': 'v1',
  'model_path': 'models/iris_model_v1.pkl',
  'metrics': {'f1_macro': 0.96},
  'description': 'baseline',
  'author': 'ml_package',
  'created_at': '2026-06-24T22:23:53.828217+00:00',
  'status': 'active',
  'artifact_sha256': None,
  'dataset_fingerprint': None,
  'parent_version': None,
  'tags': []}]

## Testing Framework

### Unit Tests
Validation, loader, prediction engine, version registry.

### Integration Tests
FastAPI routes through ASGI + HTTP client.

### API Live Tests
`requests` tests included with sandbox-aware skip when sockets are blocked.


In [4]:
import subprocess
import sys

result = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "tests/test_versioning.py", "tests/test_api.py"],
    capture_output=True,
    text=True,
    check=False,
)
print(result.stdout)
print("Exit code:", result.returncode)


........................                                                 [100%]
=============================== warnings summary ===============================
.venv/lib/python3.12/site-packages/matplotlib/_fontconfig_pattern.py:88
.venv/lib/python3.12/site-packages/matplotlib/_fontconfig_pattern.py:88
.venv/lib/python3.12/site-packages/matplotlib/_fontconfig_pattern.py:88
.venv/lib/python3.12/site-packages/matplotlib/_fontconfig_pattern.py:88
.venv/lib/python3.12/site-packages/matplotlib/_fontconfig_pattern.py:88
  /home/ahmad/AI/Github/40 AI-ML Projects for Beginners/MLOps, UI, and Deployment/Packaging Machine Learning Models/.venv/lib/python3.12/site-packages/matplotlib/_fontconfig_pattern.py:88: PyparsingDeprecationWarning: 'parseString' deprecated - use 'parse_string'
    parse = parser.parseString(pattern)

.venv/lib/python3.12/site-packages/matplotlib/_fontconfig_pattern.py:92
.venv/lib/python3.12/site-packages/matplotlib/_fontconfig_pattern.py:92
.venv/lib/python3.12/site-pack

## CLI Interface Demonstration
CLI is useful for batch scoring and automation where full API server is unnecessary.


In [5]:
import subprocess
import sys

cmd = [
    sys.executable,
    "-m",
    "ml_package.cli.predict",
    "--model-path",
    "models/iris_model.pkl",
    "predict",
    "--sepal-length",
    "5.1",
    "--sepal-width",
    "3.5",
    "--petal-length",
    "1.4",
    "--petal-width",
    "0.2",
]
out = subprocess.run(cmd, capture_output=True, text=True, check=False)
print(out.stdout)
print("CLI exit code:", out.returncode)


INFO     | Logging initialized. Log file: logs/ml_package_20260625_035357.log
{
  "prediction": 0,
  "species": "setosa",
  "confidence": 1.0,
  "probabilities": [
    1.0,
    0.0,
    0.0
  ],
  "latency_ms": 1.411,
  "model_name": "iris_classifier",
  "model_version": "v2"
}

CLI exit code: 0
